In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [12]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [13]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [14]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

In [16]:
X_train = X_train.copy()
X_test  = X_test.copy()
X_train.columns = X_train.columns.str.replace(r"\s+", "_", regex=True)
X_test.columns  = X_test.columns.str.replace(r"\s+", "_", regex=True)


# lightBoost

In [17]:
lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1 
)

model = MultiOutputRegressor(lgbm)

param_dist = {
    "estimator__n_estimators": randint(200, 1500),
    "estimator__learning_rate": uniform(0.01, 0.29), 

    "estimator__num_leaves": randint(16, 128),
    "estimator__max_depth": [-1, 3, 5, 7],
    "estimator__min_child_samples": randint(2, 30),

#     "estimator__subsample": uniform(0.6, 0.4),
#     "estimator__colsample_bytree": uniform(0.6, 0.4),

    "estimator__subsample_freq": randint(0, 8),

    "estimator__reg_alpha": uniform(0.0, 2.0),
    "estimator__reg_lambda": uniform(0.0, 5.0),
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 469
[LightGBM] [Info] Number of data points in the train set: 220, number of used features: 18
[LightGBM] [Info] Start training from score 433.158628
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

In [18]:
print(X_train.shape)
print(y_train.shape)


(220, 18)
(220, 2)
